# Silver V1 — Google Colab Runner

This notebook reads an **existing Bronze snapshot** in the correct Google Drive account and runs **Silver V1**. It does **not** ingest Bronze again or overwrite existing Silver artifacts.

Before running:
1. In Colab → **Secrets**, add `GITHUB_TOKEN` with access to the private repository and allow notebook access.
2. Sign in to the Google account containing `vietnamese-multilabel-topic-classification`.
3. Run cells sequentially. If outputs already exist, **do not rerun the publishing cell**.


In [ ]:
from google.colab import userdata
from pathlib import Path
import os
import subprocess
import tempfile

REPO_URL = "https://github.com/Ko4rut/vietnamese-multimodal-moderation.git"
BRANCH = "feat/silver-pipeline-v1"
PROJECT_DIR = Path("/content/project")

try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

if not token:
    raise ValueError("Set GITHUB_TOKEN in Colab Secrets and allow notebook access")

with tempfile.TemporaryDirectory(prefix="silver_git_") as tmp:
    askpass = Path(tmp) / "askpass.sh"
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *Password*) printf "%s\\n" "$COLAB_GITHUB_TOKEN" ;;\n'
        'esac\n',
        encoding="utf-8",
    )
    askpass.chmod(0o700)

    env = os.environ.copy()
    env.update({
        "GIT_ASKPASS": str(askpass),
        "GIT_TERMINAL_PROMPT": "0",
        "COLAB_GITHUB_TOKEN": token,
    })

    if (PROJECT_DIR / ".git").is_dir():
        dirty = subprocess.run(
            ["git", "-C", str(PROJECT_DIR), "status", "--porcelain"],
            capture_output=True, text=True, check=True, env=env,
        )
        if dirty.stdout.strip():
            raise RuntimeError("Uncommitted changes in /content/project; commit or clean before checkout")
        subprocess.run(
            ["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH],
            check=True, env=env,
        )
        subprocess.run(
            ["git", "-C", str(PROJECT_DIR), "checkout", "-B", BRANCH, "FETCH_HEAD"],
            check=True, env=env,
        )
    else:
        subprocess.run(
            ["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)],
            check=True, env=env,
        )

del token
revision = subprocess.check_output(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"], text=True
).strip()
print(f"Using branch: {BRANCH} @ {revision}")


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[data]"],
    cwd=str(PROJECT_DIR),
    check=True,
)
from vietnamese_topic_classification.data.bronze import BronzeReader
print("Python package installed; BronzeReader import OK")


In [ ]:
from google.colab import drive
from pathlib import Path

MOUNT = "/content/vtsnlp_drive"
drive.mount(MOUNT)

# This Colab session must mount the Drive account containing the 50K Bronze snapshot.
DATA_LAKE_ROOT = Path(MOUNT) / "MyDrive" / "vietnamese-multilabel-topic-classification"
MANIFEST_NAME = "source_manifest_b81fcce589_seed42_buf10000_n50000.json"

manifest_path = DATA_LAKE_ROOT / "00_source_metadata" / MANIFEST_NAME
if not manifest_path.is_file():
    raise FileNotFoundError(
        f"Bronze manifest not found: {manifest_path}. "
        "Check that the correct Google account is mounted."
    )

reader = BronzeReader(drive_root=DATA_LAKE_ROOT, manifest_name=MANIFEST_NAME)
print("Data Lake:", DATA_LAKE_ROOT)
print("Bronze snapshot:", reader.data_path.name)
print("Expected source rows:", reader.manifest["sample_size"])


In [ ]:
# Fail before processing if Silver V1 artifacts already exist.
suffix = reader.data_path.stem.removeprefix("vtsnlp_raw_sample_")
OUTPUT_FILES = {
    "silver": DATA_LAKE_ROOT / "02_silver" / f"vtsnlp_silver_v1_{suffix}.jsonl",
    "rejected": DATA_LAKE_ROOT / "02_silver" / f"vtsnlp_rejected_v1_{suffix}.jsonl",
    "report": DATA_LAKE_ROOT / "05_reports" / f"silver_quality_v1_{suffix}.json",
}
conflicts = [
    str(path) for path in OUTPUT_FILES.values()
    if path.exists() or path.with_name(path.name + ".uploading").exists()
]
if conflicts:
    raise FileExistsError(
        "Silver snapshot exists or an upload is incomplete. "
        "Do not overwrite; inspect these paths: " + ", ".join(conflicts)
    )
print("Preflight passed. No Silver V1 artifacts currently exist.")


In [ ]:
# This calls the existing CLI, which verifies Bronze and publishes Silver.
command = [
    sys.executable, str(PROJECT_DIR / "scripts/02_build_silver.py"),
    "--manifest-name", MANIFEST_NAME,
    "--data-lake-root", str(DATA_LAKE_ROOT),
]
subprocess.run(command, cwd=str(PROJECT_DIR), check=True)
print("SILVER PIPELINE COMPLETED")


In [ ]:
# Validate the outputs without re-running the pipeline.
import hashlib
import json

for artifact in OUTPUT_FILES.values():
    assert artifact.is_file(), f"Missing output: {artifact}"

report = json.loads(OUTPUT_FILES["report"].read_text(encoding="utf-8"))
assert report["source_sha256"] == reader.manifest["sha256"]
assert report["input_rows"] == reader.manifest["sample_size"]
assert report["input_rows"] == report["accepted_rows"] + report["rejected_rows"]

def inspect_jsonl(path):
    digest = hashlib.sha256()
    lines = 0
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
            lines += chunk.count(b"\n")
    return lines, digest.hexdigest()

accepted_count, accepted_sha = inspect_jsonl(OUTPUT_FILES["silver"])
rejected_count, rejected_sha = inspect_jsonl(OUTPUT_FILES["rejected"])

assert accepted_count == report["accepted_rows"]
assert rejected_count == report["rejected_rows"]
assert accepted_sha == report["silver_sha256"]
assert rejected_sha == report["rejected_sha256"]

print("SILVER VALIDATION PASSED")
print("Input rows:", report["input_rows"])
print("Accepted:", accepted_count)
print("Rejected:", rejected_count)
print("Rejection reasons:", report["rejection_reasons"])
print("Accepted domains:", len(report["accepted_domain_counts"]))
print("Silver:", OUTPUT_FILES["silver"])
print("Rejected:", OUTPUT_FILES["rejected"])
print("Report:", OUTPUT_FILES["report"])
